# MahaRainRisk: 05. Model Training, Evaluation & 2026 Validation
**Project:** Maharashtra Taluka-Level Agritech Drought Early Warning Platform  
**Purpose:** Train multi-class drought risk classifiers (Random Forest & Gradient Boosting), evaluate out-of-time predictive accuracy on the live 2026 Kharif drought test case, and export the production model.


### 1. Load Processed Master Panel


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
import joblib

current_dir = os.getcwd()
BASE_DIR = os.path.abspath(os.path.join(current_dir, '..')) if os.path.basename(current_dir) == 'notebooks' else os.path.abspath(current_dir)
master_path = os.path.join(BASE_DIR, "data", "processed", "master_panel.parquet")
df = pd.read_parquet(master_path)
print(f"Master Panel loaded: {df.shape}")


### 2. Feature Selection & Temporal Train/Test Split
Strict out-of-time validation:
- **Train Set:** Years 2018–2025 (Historical baselines including 2018 & 2023 droughts)
- **Test Set:** Year 2026 (Live active Kharif drought evaluation)


In [ ]:
feature_cols = [
    'rainfall_monthly_mm', 'rainy_days', 'temp_max_mean', 'temp_min_mean',
    'max_consecutive_dry_days', 'rain_roll_1', 'rain_roll_3', 'rain_roll_6',
    'spi_1', 'spi_3', 'spi_6',
    'normal_annual_rainfall_mm',
    'nino34_anomaly', 'oni', 'dmi',
    'district_dam_storage_pct',
    'soil_moisture_index', 'vci', 'groundwater_depth_mbgl'
]

target_col = 'drought_risk_category'

# Impute any missing values with column medians
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

# Train / Test split
train_df = df[df['year'] < 2026]
test_df = df[df['year'] == 2026]

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

print(f"Training Samples (2018-2025): {X_train.shape[0]}")
print(f"Test Samples (2026 Live Validation): {X_test.shape[0]}")


### 3. Model Training: Random Forest Multi-Class Classifier


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_split=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print("Model training complete.")


### 4. Model Evaluation on 2026 Kharif Drought Test Set


In [ ]:
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)

labels = ["Safe", "Watch", "Alert", "Severe", "Extreme"]
print("=== Classification Report (2026 Out-of-Time Test Set) ===")
print(classification_report(y_test, y_pred, target_names=labels))

macro_f1 = f1_score(y_test, y_pred, average="macro")
print(f"Macro F1-Score: {macro_f1:.4f}")


### 5. Feature Importance Analysis
Identify the top meteorological and hydrological drivers of drought in Maharashtra.


In [ ]:
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Predictive Features:")
display(importances.head(10))


### 6. Export Trained Model for Production Backend Serving


In [ ]:
ml_dir = os.path.join(BASE_DIR, "ml")
os.makedirs(ml_dir, exist_ok=True)
model_path = os.path.join(ml_dir, "drought_model.joblib")

joblib.dump({
    "model": rf_model,
    "feature_names": feature_cols,
    "labels_en": labels,
    "labels_mr": ["सुरक्षित", "लक्ष ठेवा", "सतर्कता", "तीव्र दुष्काळ", "अति तीव्र दुष्काळ"]
}, model_path)

print(f"Production model serialized to: {model_path}")
